# Data Ingestion in Qdrant vector database
Cargar:
- Markdown
- Tables
- Images
Para hacer una búsqueda híbrida

### 1 Setup enviroment

In [1]:
import os
from dotenv import load_dotenv

if load_dotenv():
    print("Cargado correctamente")


Cargado correctamente


### Qdrant

In [2]:
from qdrant_client import QdrantClient

url="http://192.168.1.103:6333"

qdrant_client = QdrantClient(
    url=url
)

d:\Cursos\Agentes\financial_deep_research_agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
#Create a collection
# from qdrant_client.models import Distance, VectorParams
# qdrant_client.create_collection(
#     collection_name="test_collection",
#     vectors_config=VectorParams(size=4, distance=Distance.DOT)
# )

In [3]:
# add vectors
from qdrant_client.models import  PointStruct

operation_info = qdrant_client.upsert(
    collection_name="test_collection",
    wait=True,
    points=[
        PointStruct(id=1, vector=[0.05, 0.61, 0.76, 0.74], payload={"city": "Berlin"}),
        PointStruct(id=2, vector=[0.19, 0.81, 0.75, 0.11], payload={"city": "London"}),
        PointStruct(id=3, vector=[0.36, 0.55, 0.47, 0.94], payload={"city": "Moscow"}),
    ]
)
operation_info

UpdateResult(operation_id=9, status=<UpdateStatus.COMPLETED: 'completed'>)

In [6]:
#run a query
search_result = qdrant_client.query_points(
    collection_name="test_collection",
    query=[0.2, 0.1, 0.9, 0.7],
    with_payload=False,
    limit=3
).points

search_result

[ScoredPoint(id=1, version=8, score=1.273, payload=None, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=3, version=8, score=1.208, payload=None, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=2, version=8, score=0.871, payload=None, vector=None, shard_key=None, order_value=None)]

In [7]:
for i in search_result:
    print(i.score)

1.273
1.208
0.871


In [8]:
# Add a filter
from qdrant_client.models import Filter, FieldCondition, MatchValue

search_result = qdrant_client.query_points(
    collection_name="test_collection",
    query=[0.2, 0.1, 0.9, 0.7],
    query_filter=Filter(
        must=[FieldCondition(key="city", match=MatchValue(value="London"))]
    ),
    with_payload=True,
    limit=3,
).points
search_result

[ScoredPoint(id=2, version=8, score=0.871, payload={'city': 'London'}, vector=None, shard_key=None, order_value=None)]

### Variables 

In [ ]:
# paths 
MARKDOWN_DIR = "D:/Cursos/Agentes/financial_deep_research_agent/data/procesed/markdown"
TABLES_DIR = "D:/Cursos/Agentes/financial_deep_research_agent/data/procesed/tables"
IMAGES_DES_DIR = "D:/Cursos/Agentes/financial_deep_research_agent/data/procesed/images_desc"

#qdrant configuration
COLLECTION_NAME = "financial_docs"
EMBEDDING_MODEL = "models/gemini-embedding-001"

In [5]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

from langchain_core.documents import Document
from langchain_qdrant import QdrantVectorStore, RetrievalMode, FastEmbedSparse

### Inicialize embeddings and client

In [7]:
 #embeddings
embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL)
sparse_embeddings = FastEmbedSparse(model_name= "Qdrant/bm25" )

2026-05-25 05:24:54.656 | WARNING  | fastembed.common.model_management:download_files_from_huggingface:223 - Local file sizes do not match the metadata.


In [12]:
result = embeddings.embed_query("hola como estas")
len(result)

3072

In [13]:
result = sparse_embeddings.embed_query("hola como estas")
result

SparseVector(indices=[723188833, 780306906, 1245366323], values=[1.0, 1.0, 1.0])

### Create or Recreate Collection

In [14]:
#create vector store
vector_store = QdrantVectorStore.from_documents(
  documents=[],
  embedding=embeddings,
  sparse_embedding=sparse_embeddings,
  url=url,
  collection_name=COLLECTION_NAME,
  retrieval_mode=RetrievalMode.HYBRID,
  force_recreate=False,
 )

In [15]:
vector_store.client.get_collections()

CollectionsResponse(collections=[CollectionDescription(name='test_collection'), CollectionDescription(name='financial_docs')])

In [16]:
def exctract_metada_from_filename(filename: str) -> dict:
    """Ectract metadata from filenama.
    Examples:
        - Amazon 10-Q Q1 2024.pdf
        - Microsoft 10-k 2023.pdf
    """
    filename = filename.replace(".pdf","").replace(".md","")
    parts = filename.split()

    return {
        "company_name": parts[0],
        "doc_type": parts[1],
        "fical_quarter": parts[2] if len(parts) == 4 else None,
        "fiscal_year": parts[-1] 
    }
exctract_metada_from_filename("apple 10-k 2023.md")

{'company_name': 'apple',
 'doc_type': '10-k',
 'fical_quarter': None,
 'fiscal_year': '2023'}

In [17]:
import hashlib
from pathlib import Path

def compute_file_has(file_path: Path):

    sha256_hash = hashlib.sha256()

    with open(file_path, 'rb') as f:
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256_hash.update(byte_block)
    return sha256_hash.hexdigest()

In [ ]:
compute_file_has(Path(r"D:\Cursos\Agentes\financial_deep_research_agent\data\procesed\markdown\apple\apple 8-k q4 2023.md"))

In [19]:
#get the list of ingested file
all_points = vector_store.client.scroll(
    collection_name=COLLECTION_NAME,
    limit=10_00,
    with_payload=True
)
all_points

([Record(id='1ab9a4c1-232c-4ed4-86ea-adc4ddf1599e', payload={'page_content': '**Metric:** Cumulative 5-year total stockholder return for a $100 investment, including reinvestment of dividends.\n\n**Key Data Points and Values (Approximate):**\n\n*   **Starting Value (12/18):** All investments began at $100.\n*   **Ending Value (12/23):**\n    *   Alphabet Inc. Class A: ~$275\n    *   S&P 500: ~$220\n    *   NASDAQ Composite: ~$240\n    *   RDG Internet Composite: ~$160\n*   **Peak Values (around 12/21):**\n    *   Alphabet Inc. Class A: ~$310\n    *   NASDAQ Composite: ~$280\n    *   RDG Internet Composite: ~$230\n    *   S&P 500: ~$200 (continued growth to 12/23)\n*   **Trough Values (around 6/22, after 12/21 peak):**\n    *   Alphabet Inc. Class A: ~$150\n    *   NASDAQ Composite: ~$150\n    *   RDG Internet Composite: ~$110\n    *   S&P 500: ~$150\n\n**Significant Trends:**\n\n*   **Overall Growth:** All investments showed positive cumulative returns over the five-year period from De

In [20]:
def get_processed_hashes():
    processed_hashes=set()
    offset=None
    
    while True:
        points, offset = vector_store.client.scroll(
            collection_name=COLLECTION_NAME,
            limit=10_000,
            with_payload=True,
            offset=offset
        )

        if not points:
            break

        processed_hashes.update(point.payload['metadata']['file_hash'] for point in points)

        if offset is None:
            break
    return processed_hashes

In [21]:
processed_hashes = get_processed_hashes()
processed_hashes

{'23921df6eff2f48f20182f3ef8aec20f419b3d397c1135f034ca13dcfaea6a8a',
 'fa021da2c823930ff4c0916d0cfc8f6e7c42059ecd5019f233f78b47a0006d9d'}

In [22]:
#exctract the page number from the file path
import re
def extract_page_number(file_path: Path):
    pattern = r'page_(\d+)'
    match = re.search(pattern=pattern, string=file_path.stem)
    return int(match.group(1)) if match else None


In [ ]:
file_path = Path(r"D:\Cursos\Agentes\financial_deep_research_agent\data\procesed\tables\amazon\amazon 10-k 2023\table_1_page_2.md")
extract_page_number(file_path)

2

### Ingestion Function

In [24]:
def ingest_file_in_db(file_path: Path, processed_hashes):
    file_hash = compute_file_has(file_path)
    if file_hash in processed_hashes:
        return print(f"El siguiente archivo ya fue subido: {file_path.name}")

    path_str = str(file_path)
    if 'markdown' in path_str:
        content_type = 'text'
        doc_name = file_path.name
    elif 'tables' in path_str:
        content_type = 'tables'
        doc_name = file_path.parent.name
    elif 'images_desc' in path_str:
        content_type = 'image'
        doc_name = file_path.parent.name
    else:
        content_type = 'unknown'
        doc_name = file_path.name

    content = file_path.read_text(encoding='utf-8')
    base_metadata = exctract_metada_from_filename(doc_name)
    base_metadata.update({
        'content_type': content_type,
        'file_hash': file_hash,
        'source_file': doc_name
    })

    if content_type == 'text':
        #write methid for ingesting markdown data
        pages = content.split('<!---page break--->')
        documents = []
        for idx, page in enumerate(pages, start=1):
            metadata = base_metadata.copy()
            metadata.update({'page':idx})
            documents.append(Document(page_content=page, metadata=metadata))
        
        vector_store.add_documents(documents)

    else:
        #write method to ingest images desc and tables .md tada
        page_num = extract_page_number(file_path)
        metadata = base_metadata.copy()
        metadata.update({'page':page_num})
        documents = [Document(page_content=content, metadata=metadata)]

        vector_store.add_documents(documents)

    processed_hashes.add(file_hash)

In [ ]:
file_path  = Path(r"D:\Cursos\Agentes\financial_deep_research_agent\data\procesed\tables\meta\meta 10-k 2023\table_7_page_61.md")
processed_hashes = get_processed_hashes()

ingest_file_in_db(file_path, processed_hashes)

El siguiente archivo ya fue subido: table_7_page_61.md


In [29]:
from tqdm import tqdm

base_path = Path("documents/")
all_md_files = list(base_path.rglob("*.md"))
# for md_file in all_md_files:
#     print(md_file)

# for md_file in tqdm(all_md_files):
#     ingest_file_in_db(md_file, processed_hashes)

In [27]:
len(all_md_files)

2008

In [30]:
count = vector_store.client.count(collection_name=COLLECTION_NAME)
print(f"Documentos en la colección: {count}")
print(f"Hashes únicos: {len(processed_hashes)}")

Documentos en la colección: count=3944
Hashes únicos: 1136


In [ ]:
from pathlib import Path

file_path = Path(r"D:\Cursos\Agentes\financial_deep_research_agent\data\procesed\tables\amazon\amazon 10-k 2023\table_7_page_43.md")
file_hash = compute_file_has(file_path)

print(f"Hash del archivo: {file_hash}")

# Buscar en la colección por ese hash
from qdrant_client.models import Filter, FieldCondition, MatchValue

results, _ = vector_store.client.scroll(
    collection_name=COLLECTION_NAME,
    scroll_filter=Filter(
        must=[FieldCondition(key="metadata.file_hash", match=MatchValue(value=file_hash))]
    ),
    with_payload=True,
    limit=10
)

print(f"Documentos encontrados: {len(results)}")
for r in results:
    print(f"  - source: {r.payload.get('metadata', {}).get('source_file')}")
    print(f"  - page: {r.payload.get('metadata', {}).get('page')}")
    print(f"  - content_type: {r.payload.get('metadata', {}).get('content_type')}")
    print(f"  - contenido: {r.payload.get('page_content', '')[:200]}")
    print()

Hash del archivo: e455f3e602f3d249ee860725d6d53f5baf14eeb06cf39c63110ac14bd16fbdce
Documentos encontrados: 1
  - source: amazon 10-k 2023
  - page: 43
  - content_type: tables
  - contenido: **Page:** 43

Basic earnings per share is calculated using our weighted-average outstanding common shares. Diluted earnings per share is calculated using our weighted-average outstanding common shares



In [37]:
info = vector_store.client.get_collection(collection_name=COLLECTION_NAME)
print(f"Vectores: {info.points_count}")
print(f"Estado: {info.status}")

Vectores: 3944
Estado: green


In [43]:
print(f"Tipos de vectores: {point.vector.keys()}")

for name, vec in point.vector.items():
    if hasattr(vec, 'indices'):
        # Sparse vector
        print(f"\n{name} (sparse):")
        print(f"  Elementos no-cero: {len(vec.indices)}")
        print(f"  Primeros 5 índices: {vec.indices[:5]}")
        print(f"  Primeros 5 valores: {vec.values[:5]}")
    else:
        # Dense vector
        print(f"\n{name} (dense):")
        print(f"  Dimensiones: {len(vec)}")
        print(f"  Primeros 5 valores: {vec[:5]}")

Tipos de vectores: dict_keys(['langchain-sparse', ''])

langchain-sparse (sparse):
  Elementos no-cero: 230
  Primeros 5 índices: [3968961, 19522071, 28501148, 35016891, 35433555]
  Primeros 5 valores: [1.0759385, 1.5516007, 1.4451215, 1.0759385, 1.959197]

 (dense):
  Dimensiones: 3072
  Primeros 5 valores: [0.0154133765, 0.021908557, 0.01607775, -0.06771266, -0.0041761585]
